## Step 1: Import Required Libraries

In [ ]:
import os
import sys
import urllib.request
import zipfile
from pathlib import Path
from tqdm import tqdm

print("✓ Libraries imported successfully")

## Step 2: Configure Dataset Paths

All paths are relative to the project directory.

In [ ]:
# Setup paths (relative to notebook location)
notebook_dir = Path.cwd()
base_dir = notebook_dir / "data"
div2k_dir = base_dir / "DIV2K"
train_dir = div2k_dir / "train"
zip_path = div2k_dir / "DIV2K_train_HR.zip"

# Create directories
div2k_dir.mkdir(parents=True, exist_ok=True)

print("Dataset Configuration:")
print(f"  Project directory: {notebook_dir}")
print(f"  Dataset directory: {div2k_dir}")
print(f"  Training images: {train_dir}")
print(f"  ZIP file: {zip_path}")

## Step 3: Check if Dataset Already Exists

In [ ]:
def check_existing_dataset(dataset_dir):
    """Check if dataset already exists and count images."""
    if not dataset_dir.exists():
        return False, 0
    
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp']
    image_files = []
    
    for ext in image_extensions:
        image_files.extend(list(dataset_dir.rglob(f'*{ext}')))
    
    return len(image_files) > 0, len(image_files)

exists, num_images = check_existing_dataset(train_dir)

if exists:
    print(f"⚠️  Dataset already exists!")
    print(f"   Location: {train_dir}")
    print(f"   Images found: {num_images}")
    print("\nSkip to Step 6 to verify, or continue to re-download.")
else:
    print("✓ No existing dataset found. Proceeding with download...")

## Step 4: Download DIV2K Dataset

**Note**: This will download ~3GB. Make sure you have enough disk space and a stable internet connection.

In [ ]:
class DownloadProgressBar(tqdm):
    """Progress bar for downloads."""
    def update_to(self, b=1, bsize=1, tsize=None):
        if tsize is not None:
            self.total = tsize
        self.update(b * bsize - self.n)

def download_file(url, output_path):
    """Download file with progress bar."""
    print(f"Downloading from: {url}")
    print(f"Saving to: {output_path}")
    
    with DownloadProgressBar(unit='B', unit_scale=True, miniters=1, desc=output_path.name) as t:
        urllib.request.urlretrieve(url, filename=output_path, reporthook=t.update_to)
    
    print(f"\n✓ Download complete!")

# Download URL
url = "http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip"

# Download if not already present
if zip_path.exists():
    print(f"✓ ZIP file already exists: {zip_path}")
    print("Skipping download.")
else:
    try:
        download_file(url, zip_path)
    except Exception as e:
        print(f"\n✗ Download failed: {e}")
        print("\nAlternative: Download manually from:")
        print("https://data.vision.ee.ethz.ch/cvl/DIV2K/")
        print(f"Save to: {zip_path}")

## Step 5: Extract Dataset

In [ ]:
def extract_zip(zip_path, extract_dir):
    """Extract ZIP file with progress bar."""
    print(f"Extracting: {zip_path}")
    print(f"Extract to: {extract_dir}")
    
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        members = zip_ref.namelist()
        with tqdm(total=len(members), desc="Extracting", unit="file") as pbar:
            for member in members:
                zip_ref.extract(member, extract_dir)
                pbar.update(1)
    
    print(f"\n✓ Extraction complete!")

if zip_path.exists():
    try:
        extract_zip(zip_path, div2k_dir)
        
        # Move extracted files to train directory
        extracted_dir = div2k_dir / "DIV2K_train_HR"
        if extracted_dir.exists() and extracted_dir != train_dir:
            print(f"\nMoving images to: {train_dir}")
            if train_dir.exists():
                import shutil
                shutil.rmtree(train_dir)
            extracted_dir.rename(train_dir)
            print("✓ Images moved successfully")
        
    except Exception as e:
        print(f"\n✗ Extraction failed: {e}")
else:
    print("✗ ZIP file not found. Please run Step 4 first.")

## Step 6: Verify Dataset

In [ ]:
def verify_dataset(dataset_dir):
    """Verify dataset and show statistics."""
    if not dataset_dir.exists():
        print(f"✗ Dataset directory not found: {dataset_dir}")
        return False
    
    # Count images by extension
    image_extensions = ['.png', '.jpg', '.jpeg', '.bmp']
    images_by_ext = {}
    
    for ext in image_extensions:
        files = list(dataset_dir.rglob(f'*{ext}'))
        if files:
            images_by_ext[ext] = len(files)
    
    total_images = sum(images_by_ext.values())
    
    if total_images == 0:
        print(f"✗ No images found in {dataset_dir}")
        return False
    
    print("=" * 60)
    print("✓ Dataset Verification Successful!")
    print("=" * 60)
    print(f"Location: {dataset_dir}")
    print(f"Total images: {total_images}")
    print(f"\nBreakdown by format:")
    for ext, count in images_by_ext.items():
        print(f"  {ext}: {count} images")
    print("=" * 60)
    
    return True

is_valid = verify_dataset(train_dir)

if is_valid:
    print("\n✓ DIV2K dataset is ready for training!")

## Step 7: Display Sample Images (Optional)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import random

# Get list of images
image_files = list(train_dir.glob('*.png')) + list(train_dir.glob('*.jpg'))

if len(image_files) > 0:
    # Show 4 random samples
    samples = random.sample(image_files, min(4, len(image_files)))
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle('DIV2K Sample Images', fontsize=16, fontweight='bold')
    
    for idx, (ax, img_path) in enumerate(zip(axes.flat, samples)):
        img = Image.open(img_path)
        ax.imshow(img)
        ax.set_title(f'{img_path.name}\n{img.size[0]}x{img.size[1]}')
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nShowing {len(samples)} sample images from {len(image_files)} total images")
else:
    print("No images found to display")

## Step 8: Cleanup (Optional)

Remove the ZIP file to save disk space (can re-download if needed).

In [ ]:
# Uncomment to remove ZIP file
# if zip_path.exists():
#     zip_path.unlink()
#     print(f"✓ Removed ZIP file: {zip_path}")
#     print(f"   Freed up ~3GB of disk space")

print("ZIP file kept for future use.")
print(f"To remove manually: {zip_path}")

## Step 9: Start Training

Now you can start training with the DIV2K dataset!

In [ ]:
print("=" * 60)
print("Ready to Train!")
print("=" * 60)
print(f"\nDataset path: {train_dir}")
print("\nTraining commands:")
print("\n1. Sanity check (recommended first):")
print(f"   python train.py --train_dir {train_dir} --sanity_mode")
print("\n2. Full training:")
print(f"   python train.py --train_dir {train_dir} --num_epochs 100")
print("\n3. Custom configuration:")
print(f"   python train.py --train_dir {train_dir} --num_epochs 100 --batch_size 8 --image_size 128")
print("\n" + "=" * 60)